In [0]:
%sql
SELECT * FROM gizmobox.silver.orders_json

1.Access elements from the JSON object

In [0]:
%sql
SELECT 
    json_values.order_id,
    json_values.customer_id,
    json_values.order_status,
    json_values.payment_method,
    json_values.total_amount,
    json_values.transaction_timestamp,
    json_values.order_date,
    json_values.items
FROM gizmobox.silver.orders_json

2. Deduplicate Array elements

Function [array_distinct](https://docs.databricks.com/aws/en/sql/language-manual/functions/array_distinct)

In [0]:
%sql
SELECT 
    json_values.order_id,
    json_values.customer_id,
    json_values.order_status,
    json_values.payment_method,
    json_values.total_amount,
    json_values.transaction_timestamp,
    json_values.order_date,
    array_distinct(json_values.items) AS items
FROM gizmobox.silver.orders_json

3. Explode Arrays (create multiple rows for arry eleements)

Fucnction [explode](https://docs.databricks.com/aws/en/pyspark/reference/functions/explode)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_orders_exploded
AS
SELECT 
    json_values.order_id,
    json_values.customer_id,
    json_values.order_status,
    json_values.payment_method,
    json_values.total_amount,
    json_values.transaction_timestamp,
    json_values.order_date,
    explode(array_distinct(json_values.items) ) AS item
FROM gizmobox.silver.orders_json

In [0]:
%sql
SELECT 
    order_id,
    customer_id,
    order_status,
    payment_method,
    total_amount,
    transaction_timestamp,
    order_date,
    item.item_id,
    item.name,
    item.price,
    item.quantity,
    item.category,
    item.details.brand,
    item.details.color
FROM tv_orders_exploded

4. Write transformed Data to silve Schema

In [0]:
%sql
CREATE TABLE gizmobox.silver.orders
AS
SELECT 
    order_id,
    customer_id,
    order_status,
    payment_method,
    total_amount,
    transaction_timestamp,
    order_date,
    item.item_id,
    item.name,
    item.price,
    item.quantity,
    item.category,
    item.details.brand,
    item.details.color
FROM tv_orders_exploded

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders;